<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Validation_on_Test_Reproducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Download the dataset from google drive

In [1]:
#bash command/script in notebook
!gdown 1fPTHJb6LvvU3THZ2wAXPH3dUI-ziN0qh

Downloading...
From: https://drive.google.com/uc?id=1fPTHJb6LvvU3THZ2wAXPH3dUI-ziN0qh
To: /content/HPVVAL25.xlsx
100% 66.0k/66.0k [00:00<00:00, 9.86MB/s]


#Train and test the model: ANN

In [2]:
import os
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score,f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" # Required for newer PyTorch versions
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    np.random.seed(seed)
    random.seed(seed)


SEED = 42
seed_everything(seed = SEED)

df = pd.read_excel("HPVVAL25.xlsx")
df = df.dropna(subset=['HPV Status'])
tobacco_mode = df['Tobacco Consumption'].mode()[0]

df['Tobacco Consumption'] = df[
    'Tobacco Consumption'
].fillna(tobacco_mode)

alcohol_mode = df['Alcohol Consumption'].mode()[0]

df['Alcohol Consumption'] = df[
    'Alcohol Consumption'
].fillna(alcohol_mode)

df = df.drop(
    columns=[
        'PatientID',
        'CenterID',
        'Task 1',
        'Task 2',
        'Task 3'
    ]
)

df = df.dropna()
print(df.shape)

df['T-stage'] = df['T-stage'].replace({
    'T0':0,
    'T1':1,
    'T2':2,
    'T3':3,
    'T4':4
})

df['N-stage'] = df['N-stage'].replace({
    'N0':0,
    'N1':1,
    'N2':2,
    'N3':3
})

df['M-stage'] = df['M-stage'].replace({'M0':0,'M1':1})

X = df[
    [
        'Age',
        'Gender',
        'Tobacco Consumption',
        'Alcohol Consumption',
        'Performance Status',
        'Relapse',
        'RFS',
        'Treatment',
        'T-stage',
        'N-stage',
        'M-stage'
    ]
]

y = df['HPV Status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

print('train sample:',y_train.value_counts(),'\n')
print('test sample:', y_test.value_counts())

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample( X_train,y_train)
print('counting sample:', pd.Series(y_train_smote).value_counts())

#fixing data imbalance with smote
X_train_smote = torch.FloatTensor(X_train_smote)
y_train_smote = torch.LongTensor(y_train_smote.to_numpy())
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor( y_test.to_numpy())

#Model Architecture
class HPVNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(11,32)
        self.fc2 = nn.Linear(32,16)
        self.fc4 = nn.Linear(16,2)
        self.relu = nn.ReLU()

    def forward(self,x):
        # print('x:', x.shape)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc4(x)
        return x

#Defining model, and optimisation function/parameters
model = HPVNet()
weights = torch.tensor([3.0,1.0], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

#Training the model
epochs = 500
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(epochs):
    model.train()
    outputs = model(X_train_smote)
    train_loss = criterion(outputs, y_train_smote)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    model.eval()

    with torch.no_grad():
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)

    if test_loss.item() < best_val_loss:

        best_val_loss = test_loss.item()
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model.pth")

    if (epoch+1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}",
            f"Best Epoch={best_epoch}"
        )

print("Best Test Loss =", best_val_loss)
print("Best Epoch =", best_epoch)


#Inferencing the mdoel
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    probabilities = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
    predicted = torch.argmax(outputs,dim=1)

results = classification_report(y_test.numpy(),predicted.numpy(),digits=4)
print('results:',results)

# Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test.numpy(),predicted.numpy())
f1 = f1_score(y_test.numpy(), predicted.numpy())

# auc = roc_auc_score(y_test.numpy(), probabilities.numpy())
y_prob = probabilities.cpu().numpy()
if y_prob.shape[1] == 2:   # Binary classification
    auc = roc_auc_score(y_test.numpy(), y_prob[:, 1])
else:                      # Multi-class classification
    auc = roc_auc_score(
        y_test.numpy(),
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

(423, 12)
train sample: HPV Status
1.0    320
0.0     18
Name: count, dtype: int64 

test sample: HPV Status
1.0    80
0.0     5
Name: count, dtype: int64
counting sample: HPV Status
1.0    320
0.0    320
Name: count, dtype: int64
Epoch 50, Train=0.4135, Test=0.9871


/tmp/ipykernel_395/1727703751.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['T-stage'] = df['T-stage'].replace({
/tmp/ipykernel_395/1727703751.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['N-stage'] = df['N-stage'].replace({
/tmp/ipykernel_395/1727703751.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silen

Epoch 100, Train=0.2620, Test=0.7621
Epoch 150, Train=0.1696, Test=0.6190
Epoch 200, Train=0.1287, Test=0.6203
Epoch 250, Train=0.1017, Test=0.6792
Epoch 300, Train=0.0806, Test=0.7357
Epoch 350, Train=0.0649, Test=0.7651
Epoch 400, Train=0.0521, Test=0.7959
Epoch 450, Train=0.0409, Test=0.8578
Epoch 500, Train=0.0316, Test=0.9293
Best Test Loss = 0.6111751198768616
Best Epoch = 172
results:               precision    recall  f1-score   support

           0     0.1429    0.6000    0.2308         5
           1     0.9688    0.7750    0.8611        80

    accuracy                         0.7647        85
   macro avg     0.5558    0.6875    0.5459        85
weighted avg     0.9202    0.7647    0.8240        85

Balanced Accuracy: 0.6875
F1-score:          0.8611
AUC:               0.8625


In [ ]:
#Archi: 11-32-16-2
Balanced Accuracy: 0.6875
F1-score:          0.8611
AUC:               0.8625

#Train and test the model: Transformer (General) for tabular/numeric data

In [11]:
import os
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score,f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" # Required for newer PyTorch versions
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    np.random.seed(seed)
    random.seed(seed)


SEED = 42
seed_everything(seed = SEED)

df = pd.read_excel("HPVVAL25.xlsx")
df = df.dropna(subset=['HPV Status'])
tobacco_mode = df['Tobacco Consumption'].mode()[0]

df['Tobacco Consumption'] = df[
    'Tobacco Consumption'
].fillna(tobacco_mode)

alcohol_mode = df['Alcohol Consumption'].mode()[0]

df['Alcohol Consumption'] = df[
    'Alcohol Consumption'
].fillna(alcohol_mode)

df = df.drop(
    columns=[
        'PatientID',
        'CenterID',
        'Task 1',
        'Task 2',
        'Task 3'
    ]
)

df = df.dropna()
print(df.shape)

df['T-stage'] = df['T-stage'].replace({
    'T0':0,
    'T1':1,
    'T2':2,
    'T3':3,
    'T4':4
})

df['N-stage'] = df['N-stage'].replace({
    'N0':0,
    'N1':1,
    'N2':2,
    'N3':3
})

df['M-stage'] = df['M-stage'].replace({'M0':0,'M1':1})

X = df[
    [
        'Age',
        'Gender',
        'Tobacco Consumption',
        'Alcohol Consumption',
        'Performance Status',
        'Relapse',
        'RFS',
        'Treatment',
        'T-stage',
        'N-stage',
        'M-stage'
    ]
]

y = df['HPV Status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

print('train sample:',y_train.value_counts(),'\n')
print('test sample:', y_test.value_counts())

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample( X_train,y_train)
print('counting sample:', pd.Series(y_train_smote).value_counts())

#fixing data imbalance with smote
X_train_smote = torch.FloatTensor(X_train_smote)
y_train_smote = torch.LongTensor(y_train_smote.to_numpy())
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor( y_test.to_numpy())

#Model Architecture
# Transformer-based model for tabular classification
class HPVNet_Transformer(nn.Module):

    def __init__(
        self,
        num_features=11,
        d_model=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=64,
        dropout=0.1,
        num_classes=2
    ):
        super().__init__()

        self.num_features = num_features
        self.d_model = d_model

        # Convert each scalar feature into a d_model-dimensional token
        self.feature_projection = nn.Linear(1, d_model)

        # Learnable embedding identifying each feature
        self.feature_embedding = nn.Parameter(
            torch.zeros(1, num_features, d_model)
        )

        # Learnable classification token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.feature_embedding, mean=0.0, std=0.02)
        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

        nn.init.xavier_uniform_(self.feature_projection.weight)
        nn.init.zeros_(self.feature_projection.bias)

    def forward(self, x):
        batch_size = x.size(0)

        # Input: [batch_size, 11]
        # Tokens: [batch_size, 11, d_model]
        x = x.unsqueeze(-1)
        x = self.feature_projection(x)

        # Distinguish Age token from Gender, Tobacco, T-stage, etc.
        x = x + self.feature_embedding

        # Add classification token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Model interactions between tabular features
        x = self.transformer(x)

        # Use the CLS token for classification
        x = self.norm(x[:, 0])

        logits = self.classifier(x)

        return logits

#Defining model, and optimisation function/parameters
model = HPVNet_Transformer()
weights = torch.tensor([3.0,1.0], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
# optimizer = torch.optim.AdamW(model.parameters(),lr=0.001, weight_decay=0.01)

#Training the model
epochs = 500
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(epochs):
    model.train()
    outputs = model(X_train_smote)
    train_loss = criterion(outputs, y_train_smote)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    model.eval()

    with torch.no_grad():
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)

    if test_loss.item() < best_val_loss:

        best_val_loss = test_loss.item()
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model_transformer.pth")

    if (epoch+1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}",
            f"Best Epoch={best_epoch}"
        )

print("Best Test Loss =", best_val_loss)
print("Best Epoch =", best_epoch)


#Inferencing the mdoel
model.load_state_dict(torch.load("best_model_transformer.pth"))
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    probabilities = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
    predicted = torch.argmax(outputs,dim=1)

results = classification_report(y_test.numpy(),predicted.numpy(),digits=4)
print('results:',results)

# Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test.numpy(),predicted.numpy())
f1 = f1_score(y_test.numpy(), predicted.numpy())

# auc = roc_auc_score(y_test.numpy(), probabilities.numpy())
y_prob = probabilities.cpu().numpy()
if y_prob.shape[1] == 2:   # Binary classification
    auc = roc_auc_score(y_test.numpy(), y_prob[:, 1])
else:                      # Multi-class classification
    auc = roc_auc_score(
        y_test.numpy(),
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

(423, 12)
train sample: HPV Status
1.0    320
0.0     18
Name: count, dtype: int64 

test sample: HPV Status
1.0    80
0.0     5
Name: count, dtype: int64
counting sample: HPV Status
1.0    320
0.0    320
Name: count, dtype: int64


/tmp/ipykernel_395/2637985554.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['T-stage'] = df['T-stage'].replace({
/tmp/ipykernel_395/2637985554.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['N-stage'] = df['N-stage'].replace({
/tmp/ipykernel_395/2637985554.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silen

Epoch 50, Train=0.3194, Test=0.7175 Best Epoch=1
Epoch 100, Train=0.1704, Test=0.5693 Best Epoch=94
Epoch 150, Train=0.1214, Test=0.4983 Best Epoch=150
Epoch 200, Train=0.0924, Test=0.4917 Best Epoch=155
Epoch 250, Train=0.0802, Test=0.5303 Best Epoch=236
Epoch 300, Train=0.0839, Test=0.4722 Best Epoch=236
Epoch 350, Train=0.0549, Test=0.5158 Best Epoch=236
Epoch 400, Train=0.0477, Test=0.5720 Best Epoch=236
Epoch 450, Train=0.0591, Test=0.5761 Best Epoch=236
Epoch 500, Train=0.0481, Test=0.6241 Best Epoch=236
Best Test Loss = 0.4496820271015167
Best Epoch = 236
results:               precision    recall  f1-score   support

           0     0.3333    0.6000    0.4286         5
           1     0.9737    0.9250    0.9487        80

    accuracy                         0.9059        85
   macro avg     0.6535    0.7625    0.6886        85
weighted avg     0.9360    0.9059    0.9181        85

Balanced Accuracy: 0.7625
F1-score:          0.9487
AUC:               0.8575


In [ ]:
Balanced Accuracy: 0.7625
F1-score:          0.9487
AUC:               0.8575

Balanced Accuracy: 0.7750
F1-score:          0.9620
AUC:               0.8375

#Archi: 11-32-16-2
Balanced Accuracy: 0.6875
F1-score:          0.8611
AUC:               0.8625

#Train and test the model: Transformer (GPT2) for tabular/numeric data

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE
from transformers import GPT2Model


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# Load and preprocess data
# ============================================================

df = pd.read_excel("HPVVAL25.xlsx")

df = df.dropna(subset=["HPV Status"])

tobacco_mode = df["Tobacco Consumption"].mode()[0]
df["Tobacco Consumption"] = (
    df["Tobacco Consumption"].fillna(tobacco_mode)
)

alcohol_mode = df["Alcohol Consumption"].mode()[0]
df["Alcohol Consumption"] = (
    df["Alcohol Consumption"].fillna(alcohol_mode)
)

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

df["T-stage"] = df["T-stage"].replace(
    {
        "T0": 0,
        "T1": 1,
        "T2": 2,
        "T3": 3,
        "T4": 4
    }
)

df["N-stage"] = df["N-stage"].replace(
    {
        "N0": 0,
        "N1": 1,
        "N2": 2,
        "N3": 3
    }
)

df["M-stage"] = df["M-stage"].replace(
    {
        "M0": 0,
        "M1": 1
    }
)

feature_columns = [
    "Age",
    "Gender",
    "Tobacco Consumption",
    "Alcohol Consumption",
    "Performance Status",
    "Relapse",
    "RFS",
    "Treatment",
    "T-stage",
    "N-stage",
    "M-stage"
]

X = df[feature_columns].copy()
y = df["HPV Status"].copy()

print("Dataset shape:", df.shape)


# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train samples:")
print(y_train.value_counts())

print("\nTest samples:")
print(y_test.value_counts())


# ============================================================
# Standardisation
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ============================================================
# SMOTE
# ============================================================

smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nTraining samples after SMOTE:")
print(pd.Series(y_train_smote).value_counts())


# ============================================================
# Convert to tensors
# ============================================================

X_train_smote = torch.tensor(
    X_train_smote,
    dtype=torch.float32
)

y_train_smote = torch.tensor(
    np.asarray(y_train_smote),
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.long
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train_smote,
    y_train_smote
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 model for numerical tabular features
# ============================================================

class HPVNetGPT2Numerical(nn.Module):

    def __init__(
        self,
        num_features=11,
        num_classes=2,
        model_name="openai-community/gpt2",
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.num_features = num_features
        self.freeze_gpt2 = freeze_gpt2

        # Load pretrained GPT-2 weights
        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False

        hidden_size = self.gpt2.config.hidden_size

        # Map each scalar value into GPT-2 embedding space
        self.feature_projection = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size)
        )

        # Learnable identity embedding for each clinical feature
        self.feature_embedding = nn.Parameter(
            torch.empty(
                1,
                num_features,
                hidden_size
            )
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        nn.init.normal_(
            self.feature_embedding,
            mean=0.0,
            std=0.02
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        # Disable GPT-2 dropout when GPT-2 is frozen
        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(self, x):
        # x: [batch_size, 11]

        # [batch_size, 11, 1]
        x = x.unsqueeze(-1)

        # [batch_size, 11, hidden_size]
        x = self.feature_projection(x)

        # Add feature identity embeddings
        x = x + self.feature_embedding

        outputs = self.gpt2(
            inputs_embeds=x,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Mean pooling over the 11 feature tokens
        pooled_output = hidden_states.mean(dim=1)

        logits = self.classifier(pooled_output)

        return logits


# ============================================================
# Model, loss and optimiser
# ============================================================

model = HPVNetGPT2Numerical(
    num_features=len(feature_columns),
    num_classes=2,
    freeze_gpt2=True
).to(device)

weights = torch.tensor(
    [3.0, 1.0],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(
            device,
            non_blocking=True
        )

        batch_y = batch_y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch_x)

        train_loss = criterion(
            outputs,
            batch_y
        )
        print('epoch:',epoch, 'loss:',train_loss.item())
        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() * batch_x.size(0)
        )

        train_sample_count += batch_x.size(0)

    average_train_loss = (
        train_loss_sum / train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch_x, batch_y in test_loader:

            batch_x = batch_x.to(
                device,
                non_blocking=True
            )

            batch_y = batch_y.to(
                device,
                non_blocking=True
            )

            test_outputs = model(batch_x)

            test_loss = criterion(
                test_outputs,
                batch_y
            )

            test_loss_sum += (
                test_loss.item() * batch_x.size(0)
            )

            test_sample_count += batch_x.size(0)

    average_test_loss = (
        test_loss_sum / test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_numerical.pth"
        )

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Inference
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_numerical.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(
            device,
            non_blocking=True
        )

        outputs = model(batch_x)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            batch_y.cpu()
        )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

Using device: cpu
Dataset shape: (423, 12)
Train samples:
HPV Status
1.0    320
0.0     18
Name: count, dtype: int64

Test samples:
HPV Status
1.0    80
0.0     5
Name: count, dtype: int64

Training samples after SMOTE:
HPV Status
1.0    320
0.0    320
Name: count, dtype: int64


/tmp/ipykernel_15519/836605538.py:95: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"] = df["T-stage"].replace(
/tmp/ipykernel_15519/836605538.py:105: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"] = df["N-stage"].replace(
/tmp/ipykernel_15519/836605538.py:114: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_si

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable parameters: 111746
epoch: 0 loss: 0.5979130864143372
epoch: 0 loss: 0.5957242846488953
epoch: 0 loss: 0.5899146199226379
epoch: 1 loss: 0.5910912156105042
epoch: 1 loss: 0.5311576724052429
epoch: 1 loss: 0.581595242023468
epoch: 2 loss: 0.5760769844055176
epoch: 2 loss: 0.5503295660018921
epoch: 2 loss: 0.5492193698883057
epoch: 3 loss: 0.5619280338287354
epoch: 3 loss: 0.554093062877655
epoch: 3 loss: 0.543767511844635
epoch: 4 loss: 0.553950846195221
epoch: 4 loss: 0.5560071468353271
epoch: 4 loss: 0.4947461187839508
epoch: 5 loss: 0.5368639826774597
epoch: 5 loss: 0.5262832641601562
epoch: 5 loss: 0.5551398396492004
epoch: 6 loss: 0.5428649187088013
epoch: 6 loss: 0.5161741971969604
epoch: 6 loss: 0.505219042301178
epoch: 7 loss: 0.4989008903503418
epoch: 7 loss: 0.5090405344963074
epoch: 7 loss: 0.5459240674972534
epoch: 8 loss: 0.4996073246002197
epoch: 8 loss: 0.5032231211662292
epoch: 8 loss: 0.4731597602367401
epoch: 9 loss: 0.5148190259933472
epoch: 9 loss: 0.4255069

In [8]:
len(train_loader), len(train_loader.dataset)

(5, 640)

#Train and test the model: Transformer (GPT2) for Text Data

Age: 54. Gender: 1. Tobacco consumption: 0. Alcohol consumption: 1.
Performance status: 0. Relapse: 0. RFS: 24. Treatment: 1.
T stage: T3. N stage: N1. M stage: M0.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from transformers import (
    GPT2Tokenizer,
    GPT2Model
)


# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ============================================================
# Load and preprocess data
# ============================================================

df = pd.read_excel("HPVVAL25.xlsx")

df = df.dropna(subset=["HPV Status"])

tobacco_mode = df["Tobacco Consumption"].mode()[0]
df["Tobacco Consumption"] = (
    df["Tobacco Consumption"].fillna(tobacco_mode)
)

alcohol_mode = df["Alcohol Consumption"].mode()[0]
df["Alcohol Consumption"] = (
    df["Alcohol Consumption"].fillna(alcohol_mode)
)

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

print("Dataset shape:", df.shape)


# ============================================================
# Normalise stage labels for readable text
# ============================================================

def format_t_stage(value):
    value = str(value)

    if value.startswith("T"):
        return value

    return f"T{int(float(value))}"


def format_n_stage(value):
    value = str(value)

    if value.startswith("N"):
        return value

    return f"N{int(float(value))}"


def format_m_stage(value):
    value = str(value)

    if value.startswith("M"):
        return value

    return f"M{int(float(value))}"


# ============================================================
# Convert one patient record into text
# ============================================================

def patient_to_text(row):
    text = (
        f"Patient clinical information. "
        f"Age: {row['Age']}. "
        f"Gender: {row['Gender']}. "
        f"Tobacco consumption: "
        f"{row['Tobacco Consumption']}. "
        f"Alcohol consumption: "
        f"{row['Alcohol Consumption']}. "
        f"Performance status: "
        f"{row['Performance Status']}. "
        f"Relapse: {row['Relapse']}. "
        f"Relapse-free survival: {row['RFS']}. "
        f"Treatment: {row['Treatment']}. "
        f"T stage: {format_t_stage(row['T-stage'])}. "
        f"N stage: {format_n_stage(row['N-stage'])}. "
        f"M stage: {format_m_stage(row['M-stage'])}. "
        f"Predict the HPV status."
    )

    return text


df["patient_text"] = df.apply(
    patient_to_text,
    axis=1
)

print("\nExample patient text:")
print(df["patient_text"].iloc[0])


# ============================================================
# Text and labels
# ============================================================

texts = df["patient_text"].tolist()

labels = (
    df["HPV Status"]
    .astype(int)
    .to_numpy()
)


# ============================================================
# Train-test split
# ============================================================

train_texts, test_texts, y_train, y_test = (
    train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=SEED,
        stratify=labels
    )
)

print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())


# ============================================================
# Tokenizer
# ============================================================

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no default padding token
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 128


# ============================================================
# Dataset
# ============================================================

class HPVTextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=128
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        encoding = self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": (
                encoding["input_ids"].squeeze(0)
            ),
            "attention_mask": (
                encoding["attention_mask"].squeeze(0)
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


train_dataset = HPVTextDataset(
    texts=train_texts,
    labels=y_train,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = HPVTextDataset(
    texts=test_texts,
    labels=y_test,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)


# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# GPT-2 text classifier
# ============================================================

class HPVNetGPT2Text(nn.Module):

    def __init__(
        self,
        model_name="openai-community/gpt2",
        num_classes=2,
        dropout=0.1,
        freeze_gpt2=True
    ):
        super().__init__()

        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False
        self.gpt2.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        hidden_size = self.gpt2.config.hidden_size

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state

        # Masked mean pooling
        expanded_mask = (
            attention_mask
            .unsqueeze(-1)
            .expand_as(hidden_states)
            .float()
        )

        hidden_sum = (
            hidden_states * expanded_mask
        ).sum(dim=1)

        valid_token_count = (
            expanded_mask.sum(dim=1)
            .clamp(min=1e-9)
        )

        pooled_output = (
            hidden_sum / valid_token_count
        )

        logits = self.classifier(
            pooled_output
        )

        return logits


# ============================================================
# Model
# ============================================================

model = HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)


# ============================================================
# Class weights
# ============================================================

class_counts = np.bincount(y_train)

class_weights = (
    len(y_train)
    / (
        len(class_counts) *
        class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# Optimiser
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# Training
# ============================================================

EPOCHS = 100
best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        train_loss = criterion(
            outputs,
            labels
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda parameter: parameter.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item() *
            input_ids.size(0)
        )

        train_sample_count += (
            input_ids.size(0)
        )

    average_train_loss = (
        train_loss_sum /
        train_sample_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_sample_count = 0

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch["label"].to(
                device,
                non_blocking=True
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss = criterion(
                outputs,
                labels
            )

            test_loss_sum += (
                test_loss.item() *
                input_ids.size(0)
            )

            test_sample_count += (
                input_ids.size(0)
            )

    average_test_loss = (
        test_loss_sum /
        test_sample_count
    )

    if average_test_loss < best_val_loss:

        best_val_loss = average_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_model_gpt2_text.pth"
        )

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch {epoch + 1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"Best Epoch={best_epoch}"
        )


print("\nBest Test Loss:", best_val_loss)
print("Best Epoch:", best_epoch)


# ============================================================
# Load best model
# ============================================================

checkpoint = torch.load(
    "best_model_gpt2_text.pth",
    map_location=device
)

model.load_state_dict(checkpoint)
model.eval()


# ============================================================
# Inference
# ============================================================

all_probabilities = []
all_predictions = []
all_targets = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["label"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu()
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            labels.cpu()
        )


probabilities = torch.cat(
    all_probabilities,
    dim=0
).numpy()

predicted = torch.cat(
    all_predictions,
    dim=0
).numpy()

y_true = torch.cat(
    all_targets,
    dim=0
).numpy()


# ============================================================
# Evaluation
# ============================================================

results = classification_report(
    y_true,
    predicted,
    digits=4,
    zero_division=0
)

print("\nClassification report:")
print(results)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:, 1]
)

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")